In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-kaizen

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT09 — Exercise 4: Advanced RAG and Evaluation
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Implement hybrid search (BM25 + vector), re-ranking, and
#   evaluate RAG quality with RAGAS-style metrics.
#
# TASKS:
#   1. Implement BM25 retrieval
#   2. Combine BM25 + vector search (hybrid)
#   3. Implement cross-encoder re-ranking
#   4. Evaluate with faithfulness, relevance, answer correctness
#   5. Compare basic vs hybrid vs re-ranked RAG
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import math
import os
import re
from collections import Counter

import polars as pl

from kaizen_agents import Delegate
from kaizen import Signature, InputField, OutputField
from kaizen_agents.agents.specialized.simple_qa import SimpleQAAgent

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

if not os.environ.get("OPENAI_API_KEY"):
    print("\u26a0 OPENAI_API_KEY not set \u2014 skipping LLM exercises.")
    print("  Set it in .env to run this exercise with real LLM calls.")
    import sys

    sys.exit(0)

model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
print(f"LLM Model: {model}")



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
regulations = loader.load("ascent09", "sg_regulations.parquet")


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """Split text into overlapping chunks."""
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if end < len(text):
            bp = max(chunk.rfind("."), chunk.rfind("\n"))
            if bp > chunk_size // 2:
                chunk = text[start : start + bp + 1]
                end = start + bp + 1
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if c]


texts = regulations.select("text").to_series().to_list()
all_chunks = []
for i, text in enumerate(texts):
    for j, chunk in enumerate(chunk_text(text)):
        all_chunks.append({"doc_idx": i, "chunk_idx": j, "text": chunk})

chunk_texts = [c["text"] for c in all_chunks]
print(f"Total chunks: {len(all_chunks)}")



## TASK 1: Implement BM25 retrieval


In [ ]:
def tokenize_simple(text: str) -> list[str]:
    """Simple whitespace + lowercase tokenizer."""
    return re.findall(r"\w+", text.lower())


class BM25:
    """BM25 ranking algorithm for text retrieval.

    __init__: tokenize all documents, compute avg_dl, build document-frequency table.
    _idf(term): log((N - df + 0.5) / (df + 0.5) + 1.0)
    score(query): for each doc, sum BM25 term scores over query tokens.
    search(query, top_k): return top_k dicts with 'idx', 'score', 'text'.
    """

    def __init__(self, documents: list[str], k1: float = 1.5, b: float = 0.75):
        # TODO: Implement BM25 __init__: store k1/b, tokenize all docs, compute
        # avg_dl, build self.df (Counter of per-term document frequency).
        ____
        ____

    def _idf(self, term: str) -> float:
        # TODO: Compute IDF: log((n_docs - df + 0.5) / (df + 0.5) + 1.0)
        ____

    def score(self, query: str) -> list[float]:
        # TODO: Tokenize query. For each document, sum BM25 scores over query terms:
        # score += idf(term) * tf*(k1+1) / (tf + k1*(1 - b + b*dl/avg_dl))
        ____
        ____

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        # TODO: Call self.score(query), sort by score descending, return top_k
        # dicts with keys 'idx', 'score', 'text'.
        ____
        ____


bm25 = BM25(chunk_texts)
test_query = "capital adequacy requirements for banks"
bm25_results = bm25.search(test_query, top_k=5)

print(f"\n=== BM25 Retrieval ===")
print(f"Query: {test_query}")
for i, r in enumerate(bm25_results[:3]):
    print(f"  [{i+1}] score={r['score']:.3f}: {r['text'][:150]}...")



## TASK 2: Combine BM25 + vector search (hybrid)


In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb) if na > 0 and nb > 0 else 0.0


async def get_embedding(text: str, delegate: Delegate) -> list[float]:
    """Generate an 8-dim pseudo-embedding via Delegate (same pattern as Ex 3)."""
    prompt = f"""Convert to 8 numbers between -1 and 1 representing:
[finance, legal, tech, compliance, sentiment, formality, specificity, complexity].
Text: "{text[:300]}"
Return ONLY 8 comma-separated numbers."""
    response = ""
    async for event in delegate.run(prompt):
        if hasattr(event, "text"):
            response += event.text
    try:
        nums = [float(x.strip()) for x in response.strip().split(",")[:8]]
        while len(nums) < 8:
            nums.append(0.0)
        return nums
    except (ValueError, IndexError):
        return [0.0] * 8


def normalize_scores(scores: list[float]) -> list[float]:
    """Min-max normalize scores to [0, 1]."""
    mn, mx = min(scores), max(scores)
    rng = mx - mn
    if rng == 0:
        return [0.5] * len(scores)
    return [(s - mn) / rng for s in scores]


async def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.5) -> list[dict]:
    """Combine BM25 and vector search with weighted fusion (alpha blends the two).
    Strategy: get BM25 scores for all docs; embed only the top-20 BM25 candidates
    for cost efficiency; compute vector scores; normalize both; fuse as
    alpha*bm25_norm + (1-alpha)*vec_norm; return top_k sorted by fused score.
    Each result dict should include 'idx', 'score', 'text', 'bm25', 'vector'.
    """
    # TODO: Implement hybrid search — BM25 scores, vector scores for top-20
    # BM25 candidates only, normalize both, fuse with alpha weighting, return top_k.
    ____
    ____
    ____
    ____


hybrid_results  = await hybrid_search(test_query, top_k=5, alpha=0.6)
print(f"\n=== Hybrid Search (alpha=0.6) ===")
for i, r in enumerate(hybrid_results[:3]):
    print(
        f"  [{i+1}] hybrid={r['score']:.3f} (bm25={r['bm25']:.3f}, vec={r['vector']:.3f})"
    )
    print(f"       {r['text'][:150]}...")



## TASK 3: Implement cross-encoder re-ranking


In [ ]:
# TODO: Define a RelevanceScore Signature with:
#   InputFields: query (str), passage (str)
#   OutputFields: relevance_score (float, 0-1), reasoning (str)
____


async def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """Re-rank candidates using LLM-based cross-encoder scoring.
    Create a SimpleQAAgent with RelevanceScore Signature (budget_usd=1.0).
    Score each candidate (passage = candidate['text'][:500]).
    Attach 'rerank_score' and 'rerank_reason' to each dict, sort descending.
    """
    # TODO: Create SimpleQAAgent with RelevanceScore, score each candidate,
    # sort by rerank_score descending, return top_k.
    ____
    ____


reranked  = await rerank(test_query, hybrid_results, top_k=3)
print(f"\n=== Re-Ranked Results ===")
for i, r in enumerate(reranked):
    print(f"  [{i+1}] rerank={r['rerank_score']:.3f}: {r['rerank_reason'][:100]}")
    print(f"       {r['text'][:150]}...")



## TASK 4: Evaluate with faithfulness, relevance, answer correctness


In [ ]:
# TODO: Define a RAGEvaluation Signature with:
#   InputFields: question (str), context (str), answer (str)
#   OutputFields: faithfulness (float, 0-1), relevance (float, 0-1),
#                 completeness (float, 0-1), evaluation_notes (str)
____


async def evaluate_rag(question: str, context: str, answer: str) -> dict:
    """Evaluate a RAG response using RAGAS-style metrics via SimpleQAAgent."""
    # TODO: Create SimpleQAAgent with RAGEvaluation Signature (budget_usd=0.5),
    # run it, return dict with keys faithfulness/relevance/completeness/notes.
    ____
    ____


async def rag_answer(query: str, results: list[dict]) -> tuple[str, str]:
    """Generate an answer from retrieved context via Delegate (budget_usd=0.5)."""
    delegate = Delegate(model=model, budget_usd=0.5)
    context = "\n\n---\n\n".join(r["text"] for r in results)
    prompt = f"Answer using ONLY the provided context.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    response = ""
    async for event in delegate.run(prompt):
        if hasattr(event, "text"):
            response += event.text
    return response.strip(), context


answer, context  = await rag_answer(test_query, reranked)
eval_result  = await evaluate_rag(test_query, context, answer)
print(f"\n=== RAG Evaluation ===")
print(f"Faithfulness: {eval_result['faithfulness']:.2f}")
print(f"Relevance:    {eval_result['relevance']:.2f}")
print(f"Completeness: {eval_result['completeness']:.2f}")
print(f"Notes: {eval_result['notes']}")



## TASK 5: Compare basic vs hybrid vs re-ranked RAG


In [ ]:
comparison = pl.DataFrame(
    {
        "method": ["BM25-only", "Hybrid (BM25+Vector)", "Hybrid + Re-ranked"],
        "retrieval_quality": [
            "Keyword match only",
            "Semantic + keyword",
            "LLM-judged relevance",
        ],
        "latency": ["Fast (no LLM)", "Medium (embeddings)", "Slow (per-candidate LLM)"],
        "best_for": ["Exact term queries", "General questions", "High-stakes answers"],
    }
)

print(f"\n=== Method Comparison ===")
print(comparison)
print(f"\nProduction recommendation:")
print(f"  1. BM25 for fast first-pass retrieval")
print(f"  2. Vector search for semantic expansion")
print(f"  3. Re-ranking only on top-N candidates (cost control)")
print(f"  4. Always evaluate with faithfulness + relevance metrics")

print("\n✓ Exercise 4 complete — hybrid RAG with BM25, re-ranking, and evaluation")

